# Preview SNI 300 g dari instance-crop v1

Notebook ini hanya membuat dan menampilkan data sintetis. **Tidak ada training.**

Empat preview dibuat: `source_empirical/A1`, `source_empirical/A2`, `defect_enriched/A1`, dan `defect_enriched/A2`. Normal tetap menjadi kelas terbanyak. `source_empirical` bukan klaim prevalensi dunia nyata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, subprocess, sys

REPO_ROOT = Path('/content/coffee-bean-detection')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--branch', 'agent/add-vadcp-pipeline', 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
SRC_ROOT = REPO_ROOT / 'src'
os.environ['PYTHONPATH'] = str(SRC_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()
import coffee_detector
print('IMPORT BERHASIL:', coffee_detector.__file__)

In [ ]:
CROP_DATASET_ROOT = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
OUTPUT_ROOT = Path('/content/coffee-sni-300g-preview-v2')
DRIVE_RESULT_ROOT = Path('/content/drive/MyDrive/coffee-bean-detection/sni-crop-300g-preview')

required = [CROP_DATASET_ROOT / 'manifest.csv', CROP_DATASET_ROOT / 'complete.json', CROP_DATASET_ROOT / 'shards']
for path in required:
    print(('ADA   ' if path.exists() else 'HILANG'), '-', path)
assert all(path.exists() for path in required), 'Dataset crop di Drive belum lengkap.'
print('OUTPUT LOKAL:', OUTPUT_ROOT)
print('TRAINING: TIDAK DIJALANKAN')

In [ ]:
# Runner dipanggil langsung dari kernel; tidak memakai subprocess Python.
import shutil
from coffee_detector.run_sni_crop_preview import run_sni_crop_preview

summary_path = OUTPUT_ROOT / 'preview_summary.json'
if OUTPUT_ROOT.exists() and not summary_path.is_file():
    print('Output parsial lokal dibersihkan; cache shard tetap dipertahankan.', flush=True)
    shutil.rmtree(OUTPUT_ROOT)

if summary_path.is_file():
    print('PREVIEW SUDAH SELESAI, hasil digunakan ulang:', summary_path, flush=True)
else:
    print('MULAI PREVIEW — BUKAN TRAINING', flush=True)
    run_sni_crop_preview(
        crop_dataset_root=CROP_DATASET_ROOT,
        output_root=OUTPUT_ROOT,
        images=4,
        seed=42,
        objects_min=220,
        objects_max=300,
        canvas_size=1024,
        enriched_normal_fraction=0.55,
        max_normal_assets=220,
        max_defect_assets_per_class=25,
        shard_cache_root='/content/coffee-sni-shard-cache',
    )
    print('PREVIEW SELESAI', flush=True)

In [ ]:
import json
from IPython.display import Image as DisplayImage, display
summary = json.loads((OUTPUT_ROOT / 'preview_summary.json').read_text())
print('=== KOMPOSISI AKTUAL ===')
for arm, row in summary['compositions'].items():
    print(f"{arm:28s} normal={row['normal_fraction']:.2%} instances={row['labeled_instances']} repeated={row['repeated_assets']}")
print('\n=== CUTOUT EDGE AUDIT ===')
display(DisplayImage(filename=summary['cutout_contact_sheet']))
for arm, path in summary['raw_contact_sheets'].items():
    print('\n', arm)
    display(DisplayImage(filename=path))

In [ ]:
# Simpan ringkasan dan seluruh contact sheet ke Drive setelah review.
# Path diambil dari preview_summary.json agar tidak bergantung pada nama file manual.
import json
import shutil

summary_source = OUTPUT_ROOT / 'preview_summary.json'
summary = json.loads(summary_source.read_text())
DRIVE_RESULT_ROOT.mkdir(parents=True, exist_ok=True)

artifact_groups = {
    'cutout_contact_sheet': {'cutout': summary['cutout_contact_sheet']},
    'raw_contact_sheets': summary['raw_contact_sheets'],
    'overlay_contact_sheets': summary['overlay_contact_sheets'],
}
saved_summary = dict(summary)
for field, artifacts in artifact_groups.items():
    saved_paths = {}
    for name, source_path in artifacts.items():
        source = Path(source_path).resolve()
        if not source.is_file():
            raise FileNotFoundError(f'Artefak hasil preview tidak ditemukan: {source}')
        relative = source.relative_to(OUTPUT_ROOT.resolve())
        target = DRIVE_RESULT_ROOT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        saved_paths[name] = str(target)
        print('SAVED:', target)
    saved_summary[field] = (
        saved_paths['cutout'] if field == 'cutout_contact_sheet' else saved_paths
    )

summary_target = DRIVE_RESULT_ROOT / 'preview_summary.json'
summary_target.write_text(json.dumps(saved_summary, indent=2, ensure_ascii=False))
print('SAVED:', summary_target)
print('Training tetap belum dijalankan.')